In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

config = {
    "toImageButtonOptions": {
        "format": "png",  # Ensure the format is PNG
        "filename": "high_res_plot",
        "height": 500,
        "width": 700,
        "scale": 3,  # Multiplies resolution/DPI (e.g., 2 or 3 for crisp images)
    }
}



# Metadata observations

## Load data

In [ ]:
df = pd.read_csv("outputs/2.0-discovery_results.csv")
df

In [ ]:
print("Metadata observations CSV:")
for x in df.columns:
    print("- ", x)

## Productivity

In [ ]:
obs_count = df.groupby("snapshot_id").agg(obs_count=("snapshot_id", "count"))

print("Number of snapshots:", df["snapshot_id"].nunique())
print("Number of metadata observations:", len(df))
print("Mean observations per snapshot:", obs_count.mean().values[0])
print("Median:", obs_count.median().values[0])
print("Std dev:", obs_count.std().values[0])
print("Min:", obs_count.min().values[0])
print("Max:", obs_count.max().values[0])

In [ ]:
fig = px.histogram(
    obs_count,
    x="obs_count",
    width=600,
    height=400,
    title="Count of suggested fields per snapshot",
)
fig.show()

In [ ]:
print(obs_count["obs_count"].value_counts().sort_index().to_markdown())

Note: Take into account that the prompt specifically says "Limit the output to the 10–15 most useful reusable metadata fields. Do not attempt to exhaustively enumerate every possible field."

## Discovery diversity

In [ ]:
field_count = (
    df.groupby("metadata_field")
    .agg(count=("snapshot_id", "nunique"))
    .sort_values("count", ascending=False)
)

print("Number of unique suggested metadata fields:", df["metadata_field"].nunique())
print("Mean snapshot count per suggested metadata field:", field_count.mean().values[0])
print("Median:", field_count.median().values[0])
print("Std dev:", field_count.std().values[0])
print("Min:", field_count.min().values[0])
print("Max:", field_count.max().values[0])

In [ ]:
fig = px.histogram(
    field_count,
    # x="count",
    width=800,
    height=400,
    title="Distribution of snapshot count by suggested metadata field",
    nbins=100
)
fig.show()

In [ ]:
bins = [0, 1, 2, 3, 4, 5, 10, 15, 20, 50, 100, 150, 200, 1000]
field_count['range'] = pd.cut(field_count['count'], bins=bins)
grouped_freq = field_count['range'].value_counts().sort_index()
print(grouped_freq.to_markdown())

In [ ]:
print("Singletons:", len(field_count[field_count["count"] == 1]))

print("\nTop-k coverage")
for x in [10, 20, 50, 100]:
    val = field_count.sort_values("count", ascending=False).head(x)["count"].sum()
    pct = 100 * val / field_count["count"].sum()
    print(f"Top {x} fields: {val} ({pct:.2f}%)")

field_count["k"] = range(1, len(field_count) + 1)
field_count["coverage"] = field_count["count"].cumsum()

px.line(field_count, x="k", y="coverage", width=800, height=400, title="Top-k coverage")

In [ ]:
field_count["coverage_pct"] = 100.0 * field_count["coverage"] / 3041

print("Vocabulary concentration")
for x in [25, 50, 75, 90]:
    k = field_count[field_count["coverage_pct"] > x]["k"].min()
    cov = field_count[field_count["coverage_pct"] > x]["coverage_pct"].min()
    print(f"Top {k} fields cover {cov:.2f}% of observations")

# 1. Create your base line graph
fig = px.line(
    field_count,
    x="k",
    y="coverage_pct",
    width=600,
    height=400,
    title="Cumulative observation coverage",
    labels={
        "coverage_pct": "Observations covered (%)",
        "k": "# of discovered fields",
    },
)

xx = 20
yy = field_count.loc[field_count["k"] == 20, "coverage_pct"].values[0]

# 2. Add vertical dotted line segment (from y=0 to y=50.81)
fig.add_shape(
    type="line",
    x0=xx, y0=0,
    x1=xx, y1=yy,
    line=dict(dash="dot", color="gray", width=1.5)
)

# 3. Add horizontal dotted line segment (from x=0 to x=20)
fig.add_shape(
    type="line",
    x0=0, y0=yy,
    x1=xx, y1=yy,
    line=dict(dash="dot", color="gray", width=1.5)
)

# 4. Add the tall, wrapped text box annotation
fig.add_annotation(
    x=xx,
    y=yy,
    # <br> tags break the text to make the box narrower and taller
    text=f"Top 20 fields<br>account for<br>{yy:.2f}% of<br>observations",
    align="center", # Centers the wrapped text inside the box
    showarrow=True,
    arrowhead=2,
    ax=80,    # Shifts text box 50 pixels right
    ay=30,   # Shifts text box 50 pixels up
    bordercolor="black",
    borderwidth=1,
    borderpad=6, # Adds padding inside the box to frame the wrapped text
    bgcolor="white",
    opacity=0.9
)

fig.show(config=config)

## Corpus comparison

In [ ]:
for s in ["unhcr", "prwp", "refugee"]:
    print(s)
    print("Number of observations:", df[df["source"] == s]["metadata_field"].count())
    print("Mean observation count per snapshot:", df[df["source"] == s].groupby("snapshot_id").agg(count=("snapshot_id", "count")).mean().values[0])
    print("Number of unique suggested metadata fields:", df[df["source"] == s]["metadata_field"].nunique())

    field_count_ = df[df["source"] == s].groupby("metadata_field").agg(count=("snapshot_id", "nunique"))
    print("Mean snapshot count per suggested metadata field:", field_count_.mean().values[0])
    print("Median snapshot count per suggested metadata field:", field_count_.median().values[0])
    print("Std dev snapshot count per suggested metadata field:", field_count_.std().values[0])
    print("Min snapshot count per suggested metadata field:", field_count_.min().values[0])
    print("Max snapshot count per suggested metadata field:", field_count_.max().values[0])

    print("---")

In [ ]:
# Vocabulary overlap
voc_unhcr = set(df[df["source"] == "unhcr"]["metadata_field"].unique())
voc_prwp = set(df[df["source"] == "prwp"]["metadata_field"].unique())
voc_refugee = set(df[df["source"] == "refugee"]["metadata_field"].unique())

print("Pairwise jaccard")
print("UNHCR and PRWP:", len(voc_unhcr & voc_prwp) / len(voc_unhcr | voc_prwp))
print("UNHCR and Refugee:", len(voc_unhcr & voc_refugee) / len(voc_unhcr | voc_refugee))
print("PRWP and Refugee:", len(voc_refugee & voc_prwp) / len(voc_refugee | voc_prwp))

print("\nTriple intersection:", len(voc_unhcr & voc_prwp & voc_refugee))

print("\nOnly in UNHCR:", len(voc_unhcr - voc_prwp - voc_refugee))
print("Only in PRWP:", len(voc_prwp - voc_unhcr - voc_refugee))
print("Only in Refugee:", len(voc_refugee - voc_unhcr - voc_prwp))

## Figure vs Table

In [ ]:
for s in ["figure", "table"]:
    print(s)
    print("Number of observations:", df[df["snapshot_type"] == s]["metadata_field"].count())
    print("Mean observation count per snapshot:", df[df["snapshot_type"] == s].groupby("snapshot_id").agg(count=("snapshot_id", "count")).mean().values[0])
    print("Number of unique suggested metadata fields:", df[df["snapshot_type"] == s]["metadata_field"].nunique())

    field_count_ = df[df["snapshot_type"] == s].groupby("metadata_field").agg(count=("snapshot_id", "nunique"))
    print("Mean snapshot count per suggested metadata field:", field_count_.mean().values[0])
    print("Median snapshot count per suggested metadata field:", field_count_.median().values[0])
    print("Std dev snapshot count per suggested metadata field:", field_count_.std().values[0])
    print("Min snapshot count per suggested metadata field:", field_count_.min().values[0])
    print("Max snapshot count per suggested metadata field:", field_count_.max().values[0])

    print("---")

In [ ]:
# Vocabulary overlap
voc_figure = set(df[df["snapshot_type"] == "figure"]["metadata_field"].unique())
voc_table = set(df[df["snapshot_type"] == "table"]["metadata_field"].unique())

print("Pairwise jaccard:", len(voc_figure & voc_table) / len(voc_figure | voc_table))
print("Intersection:", len(voc_figure & voc_table))

print("\nOnly in figure:", len(voc_figure - voc_table))
print("Only in table:", len(voc_table - voc_figure))

In [ ]:
print("Top 10 figure fields:")
print(df[df["snapshot_type"] == "figure"]["metadata_field"].value_counts().head(10).to_markdown())

print("\nTop 10 table fields:")
print(df[df["snapshot_type"] == "table"]["metadata_field"].value_counts().head(10).to_markdown())

## Vocabulary accumulation curve

In [ ]:
acc_fields = set()
acc_snapshot = set()
snapshots = df["snapshot_id"].unique().tolist()
# np.random.shuffle(snapshots)

acc_curve = {}  # n_unique_snapshots: n_unique_fields
for x in snapshots:
    acc_snapshot.add(x)
    acc_fields.update(df[df["snapshot_id"] == x]["metadata_field"].unique().tolist())

    acc_curve[len(acc_snapshot)] = len(acc_fields)

acc_curve_df = (
    pd.Series(acc_curve)
    .rename("unique_metadata_field_count")
    .to_frame()
    .reset_index(names="unique_snapshot_count")
)

In [ ]:
fig = px.line(
    acc_curve_df,
    x="unique_snapshot_count",
    y="unique_metadata_field_count",
    height=400,
    width=600,
    title="Vocabulary accumulation curve",
    labels={
        "unique_metadata_field_count": "Unique metadata field count",
        "unique_snapshot_count": "# of snapshots processed",
    },
)

fig.show(config=config)

In [ ]:
print(acc_curve_df.iloc[::20].to_markdown(index=False))

In [ ]:
print(acc_curve_df.tail(11).iloc[::2].to_markdown(index=False))

In [ ]:
acc_curve_df["new_discovered_fields"] = acc_curve_df["unique_metadata_field_count"].diff()

print("Average discovery rate per snapshot:", acc_curve_df["new_discovered_fields"].mean())

Even after shuffling the dataset 5 times, average discovery rate is around 3.92.

In [ ]:
print(acc_curve_df.head(10).to_markdown(index=False))

In [ ]:
acc_curve_df["rolling_mean_5"] = acc_curve_df["new_discovered_fields"].rolling(window=5).mean()
acc_curve_df.head(10)

In [ ]:
acc_curve_df.tail(10)

In [ ]:
acc_curve_df["rolling_mean_5"].describe()

## First appearance analysis

In [ ]:
appearance_dict = {}  # metadata_field: nth_observation
appearance_container = set()

i = 0
snapshot_id = None
for _, row in df.iterrows():
    if snapshot_id != row["snapshot_id"]:
        i += 1
        snapshot_id = row["snapshot_id"]

    f = row["metadata_field"]
    if f in appearance_container:
        continue

    appearance_dict[f] = i
    appearance_container.add(f)

appearance_df = (
    pd.Series(appearance_dict)
    .rename("nth_snapshot")
    .to_frame()
    .reset_index(names="metadata_field")
)
appearance_df

In [ ]:
appearance_df_hist = appearance_df.groupby("nth_snapshot").agg(count=("metadata_field", "count"))
appearance_df_hist

In [ ]:
print(appearance_df_hist.head(10).to_markdown())

In [ ]:
# TODO: Overlay 5-point rolling average
px.bar(
    appearance_df_hist.reset_index().rename(columns={"count": "count_of_new_fields"}),
    x="nth_snapshot",
    y="count_of_new_fields",
    title="Novel metadata field discovery throughout sequential sampling",
    width=1000,
    height=400,
)

## Corpus breadth

In [ ]:
breadth_dict = {}  # metadata_field: n_corpus
for f in df["metadata_field"].unique().tolist():
    b = (f in voc_unhcr) + (f in voc_prwp) + (f in voc_refugee)
    breadth_dict[f] = b

In [ ]:
breadth_df = (
    pd.Series(breadth_dict)
    .rename("corpus_count")
    .to_frame()
    .reset_index(names="metadata_field")
)
print(breadth_df.groupby("corpus_count").agg(count=("corpus_count", "count")).to_markdown())

In [ ]:
common_fields = breadth_df[breadth_df["corpus_count"] == 3]["metadata_field"].tolist()
print(df[df["metadata_field"].isin(common_fields)]["metadata_field"].value_counts().head(20).to_markdown())

In [ ]:
single_fields = breadth_df[breadth_df["corpus_count"] == 1]["metadata_field"].tolist()
single_fields_count = (
    df[df["metadata_field"].isin(single_fields)]
    .groupby("metadata_field")
    .agg(count=("metadata_field", "count"))
)

single_fields_count[single_fields_count["count"] == 1].head(20)

# Field profiles

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

config = {
    "toImageButtonOptions": {
        "format": "png",  # Ensure the format is PNG
        "filename": "high_res_plot",
        "height": 500,
        "width": 700,
        "scale": 3,  # Multiplies resolution/DPI (e.g., 2 or 3 for crisp images)
    }
}



## Load data

In [ ]:
df = pd.read_csv("outputs/3.0-field_profiles.csv")
df

In [ ]:
from ast import literal_eval

sample_profiles = df[
    [
        "metadata_field",
        "count",
        "top_observed_values",
        "top_description_values",
        "top_reasoning_values",
        "corpora_count",
        "figure_count",
        "table_count",
    ]
].head(3)
cols = ["top_observed_values", "top_description_values", "top_reasoning_values"]
for c in cols:
    sample_profiles[c] = sample_profiles[c].apply(lambda x: ", ".join([y for y in literal_eval(x)]))
sample_profiles = sample_profiles.rename(
    columns={
        "metadata_field": "Candidate metadata field",
        "count": "Count",
        "top_observed_values": "Top observed values",
        "top_description_values": "Top descriptions",
        "top_reasoning_values": "Top reasoning",
        "corpora_count": "Number of corpus represented",
        "figure_count": "Figure count",
        "table_count": "Table count"
    }
)
sample_profiles.to_clipboard(index=False)

## Aggregation compression

In [ ]:
print(f"Compression ratio: 833 / 3041 ({(100 * 833/3041):.2f}%) ")
print(f"Mean field count per profile:", df["count"].mean())
print(f"Median field count per profile:", df["count"].median())
print(f"Std field count per profile:", df["count"].std())
print(f"Min field count per profile:", df["count"].min())
print(f"Max field count per profile:", df["count"].max())

In [ ]:
px.histogram(
    df, x="support", nbins=100, height=400, width=800, title="Support distribution"
)

In [ ]:
bins = [0, 1, 5, 10, 15, 20, 25, 50, 75, 100, 125, 150, 200, 1000]
df['n_snapshots_range'] = pd.cut(df['n_snapshots'], bins=bins)
grouped_freq = df['n_snapshots_range'].value_counts().sort_index()
print(grouped_freq.to_markdown())

In [ ]:
df["support_range"] = pd.qcut(df["support"], q=30, duplicates="drop")
grouped_freq = df["support_range"].value_counts().sort_index()
print(grouped_freq.to_markdown())

## Support distribution

In [ ]:
print(df.sort_values("count", ascending=False).head(20)[["metadata_field", "count", "support"]].to_markdown(index=False))

In [ ]:
px.histogram(df, x="support", nbins=20)

In [ ]:
df = df.sort_values("support", ascending=False)
df["rank"] = range(1, len(df) + 1)

px.line(df, x="rank", y="support", height=400, width=800, title="Support vs rank")

In [ ]:
print(df[["rank", "support"]].iloc[::50].to_markdown(index=False))

In [ ]:
cols = [
    "metadata_field",
    "n_snapshots",
    "support",
    "corpora_count",
    "figure_count",
    "table_count",
    "source_snapshot_count",
    "source_document_count",
    "source_both_count",
    "top_observed_values",
    "n_unique_observed_values",
    "top_description_values",
    "top_reasoning_values",
]
print("5 samples of support = 1:")
print(df[df["n_snapshots"] == 1][cols].sample(5).to_markdown(index=False))

In [ ]:
print("5 samples of support = 10:")
print(df[df["n_snapshots"] == 10][cols].sample(5).to_markdown(index=False))

In [ ]:
print("support = 20:")
print(df[df["n_snapshots"] == 20][cols].to_markdown(index=False))

## Corpus breadth

In [ ]:
print(df["corpora_count"].value_counts().sort_index().to_markdown())

In [ ]:
mask1 = df["figure_count"] > 0
mask2 = df["table_count"] == 0
print("Figure only:", len(df[mask1 & mask2]))

mask1 = df["figure_count"] == 0
mask2 = df["table_count"] > 0
print("Table only:", len(df[mask1 & mask2]))

mask1 = df["figure_count"] > 0
mask2 = df["table_count"] > 0
print("Figure only:", len(df[mask1 & mask2]))

## Profile richness

At this stage, only the top 3 observed values, descriptions, and reasonings were kept.

Do we want to compare the profiles against the previous unaggregated suggested metadata fields?

## Representative evidence

In [ ]:
print(df.head(3)[cols].to_markdown(index=False))

## Aggregation quality

In [ ]:
# with pd.option_context('display.max_colwidth', None):
#     display(df.head(20)[["metadata_field", "top_observed_values", "top_description_values"]])

print(df.head(20)[["metadata_field", "top_observed_values", "top_description_values"]].to_markdown(index=False))

No issues with aggregation. I think there is little room for confusion for the high support metadata_fields.

## Corpus-specific profiles

In [ ]:
print(df[df["corpora_count"] == 1].sort_values("n_snapshots", ascending=False).head(20)[cols].to_markdown(index=False))

## Figure vs table breadth

In [ ]:
mask1 = df["figure_count"] > 0
mask2 = df["table_count"] == 0
print("Top figure only:")
print(df[mask1 & mask2][cols].head(10).to_markdown(index=False))

print()

mask1 = df["figure_count"] == 0
mask2 = df["table_count"] > 0
print("Top table only:")
print(df[mask1 & mask2][cols].head(10).to_markdown(index=False))

print()

mask1 = df["figure_count"] > 0
mask2 = df["table_count"] > 0
print("Top both:")
print(df[mask1 & mask2][cols].head(10).to_markdown(index=False))

## Aggregation before ontology

In [ ]:
print(df["n_snapshots"].value_counts().sort_index().to_markdown())

## Support vs richness

In [ ]:
raw = pd.read_csv("outputs/2.0-discovery_results.csv")
richness = (
    raw.groupby("metadata_field")
    .agg(
        support=("metadata_field", "count"),
        n_unique_obs_values=("observed_value", "nunique"),
    )
    .reset_index()
)

px.scatter(
    richness,
    x="support",
    y="n_unique_obs_values",
    height=400,
    width=800,
    title="Support vs richness",
)

In [ ]:
print("Top 10 support:")
print(richness.sort_values("support", ascending=False).head(10).to_markdown(index=False))

In [ ]:
print("Top 10 number of unique observed values:")
print(richness.sort_values("n_unique_obs_values", ascending=False).head(10).to_markdown(index=False))

I think at certain parts of the scatter plot, the number of unique observations is directly proportional to the support. For 1 support fields, obviously it's 1 unique observation value. 

# Ontology v0

In [ ]:
df = pd.read_csv("outputs/3.1-ontology_v0.md", sep="|", skipinitialspace=True).dropna(
    axis=1,
    how="all",
)
df.columns = df.columns.str.strip()
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
df = df.drop(index=0)

df

## Ontology induction summary

In [ ]:
print("Number of ontology concepts:", len(df))
print(
    f"Compression ratio: {len(df)}/833 ({(100 * len(df)/833):.2f}%)",
)

I cannot calculate the aggregate field profiles per ontology concept because I do not have the labeling of the field profiles using ontology_v0. I only have that for the final schema. If we really need this, I can run the same pipeline where I would pass each profile to GPT-5.4-mini.

From ontology_v0, our next step was human refinement (keep, merge, remove).

Therefore, the succeeding analyses could not be performed.